# Fine-tune LFM2.5-VL-450M for Satellite Triage

**Kaggle GPU runtime required** (T4 or P100). Settings → Accelerator → GPU T4 x2.

This notebook fine-tunes the Liquid AI VLM on satellite imagery to produce triage JSON.

In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets huggingface_hub pillow

In [ ]:
import os
from huggingface_hub import login, hf_hub_download

# Set your HF token in Kaggle Secrets (Add-ons → Secrets → HF_TOKEN)
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
HF_TOKEN = secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)

In [ ]:
# Download training data from HuggingFace
from huggingface_hub import snapshot_download

snapshot_download(
    "marcelo-earth/VRSBench-satellite-triage-labels",
    repo_type="dataset",
    local_dir="/kaggle/working/labels",
)
!ls /kaggle/working/labels/

In [ ]:
# Download VRSBench images
import zipfile

zip_path = hf_hub_download("xiang709/VRSBench", "Images_train.zip", repo_type="dataset")
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall("/kaggle/working/images")
!ls /kaggle/working/images/ | head -5
!ls /kaggle/working/images/Images_train/ | wc -l

In [ ]:
# Build balanced training dataset (MEDIUM vs LOW+SKIP)
import json, random
from collections import defaultdict

# Load captions and labels
with open("/kaggle/working/labels/captions.jsonl") as f:
    captions = [json.loads(l) for l in f]
with open("/kaggle/working/labels/labels.jsonl") as f:
    labels = [json.loads(l) for l in f]

assert len(captions) == len(labels)
print(f"Total samples: {len(captions)}")

# Build SFT samples grouped by priority
SYSTEM_PROMPT = """You are a satellite image triage system. Analyze the image and respond ONLY with a JSON object. No other text.

Priority: CRITICAL (disasters, fires, floods), HIGH (deforestation, unusual activity, anomalies), MEDIUM (routine urban, agriculture), LOW (featureless desert, barren terrain), SKIP (heavy clouds >80%, empty ocean, image artifacts).

If the image is mostly white/bright with no ground features visible, it is cloud-covered — mark SKIP."""

USER_PROMPT = "Triage this satellite image. Respond with JSON only."

by_priority = defaultdict(list)
img_dir = "/kaggle/working/images/Images_train"

for cap, lab in zip(captions, labels):
    img_path = f"{img_dir}/{cap['image']}"
    if not os.path.exists(img_path):
        continue
    triage_json = json.dumps({
        "description": cap["caption"],
        "priority": lab["priority"],
        "reasoning": lab["reasoning"],
        "categories": lab["categories"],
    })
    sample = {
        "messages": [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
            {"role": "user", "content": [
                {"type": "image", "image": img_path},
                {"type": "text", "text": USER_PROMPT},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": triage_json}]},
        ]
    }
    by_priority[lab["priority"]].append(sample)

for p, samples in sorted(by_priority.items(), key=lambda x: -len(x[1])):
    print(f"  {p}: {len(samples)}")

In [ ]:
# Balance: downsample MEDIUM to match LOW+SKIP count
random.seed(42)

low_skip = by_priority["LOW"] + by_priority["SKIP"]
n = len(low_skip)
med_sample = random.sample(by_priority["MEDIUM"], n)

train_data = low_skip + med_sample
random.shuffle(train_data)

# 90/10 split
split = int(len(train_data) * 0.9)
train_split = train_data[:split]
eval_split = train_data[split:]

print(f"Train: {len(train_split)}, Eval: {len(eval_split)}")

# Save
with open("/kaggle/working/train.jsonl", "w") as f:
    for s in train_split:
        f.write(json.dumps(s) + "\n")
with open("/kaggle/working/eval.jsonl", "w") as f:
    for s in eval_split:
        f.write(json.dumps(s) + "\n")

In [ ]:
import torch
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, get_peft_model

BASE_MODEL = "LiquidAI/LFM2.5-VL-450M"

print("Loading model...")
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL, dtype=torch.bfloat16, device_map="auto"
)
processor = AutoProcessor.from_pretrained(BASE_MODEL)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2", "in_proj", "linear_1", "linear_2"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import torch.nn.functional as F


class TriageDataset(Dataset):
    def __init__(self, jsonl_path, processor):
        with open(jsonl_path) as f:
            self.samples = [json.loads(l) for l in f]
        self.processor = processor

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        messages = sample["messages"]

        # Extract image path
        img_path = None
        for msg in messages:
            for c in msg["content"]:
                if isinstance(c, dict) and c.get("type") == "image":
                    img_path = c["image"]
        image = Image.open(img_path).convert("RGB")

        # Build messages with actual image for processor
        proc_messages = []
        for msg in messages:
            new_content = []
            for c in msg["content"]:
                if isinstance(c, dict) and c.get("type") == "image":
                    new_content.append({"type": "image", "image": image})
                else:
                    new_content.append(c)
            proc_messages.append({"role": msg["role"], "content": new_content})

        # Tokenize full conversation (including assistant response)
        inputs = self.processor.apply_chat_template(
            proc_messages,
            add_generation_prompt=False,
            return_tensors="pt",
            return_dict=True,
            tokenize=True,
        )
        input_ids = inputs["input_ids"].squeeze(0)
        # Labels = input_ids shifted (standard causal LM)
        inputs["labels"] = input_ids.clone()
        return {k: v.squeeze(0) if v.dim() > 1 else v for k, v in inputs.items()}


train_dataset = TriageDataset("/kaggle/working/train.jsonl", processor)
eval_dataset = TriageDataset("/kaggle/working/eval.jsonl", processor)
print(f"Train: {len(train_dataset)}, Eval: {len(eval_dataset)}")
# Quick sanity check
sample = train_dataset[0]
print(f"Keys: {list(sample.keys())}")
print(f"input_ids shape: {sample['input_ids'].shape}")

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import torch.nn.functional as F


class VLMTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        # Shift: predict next token
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = F.cross_entropy(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
            ignore_index=-100,
        )
        return (loss, outputs) if return_outputs else loss


training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    learning_rate=1e-4,
    warmup_steps=50,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    bf16=True,
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    remove_unused_columns=False,
    report_to="none",
)

trainer = VLMTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=DataCollatorForSeq2Seq(processor.tokenizer, padding=True),
)

trainer.train()

In [ ]:
# Merge LoRA and save
merged = model.merge_and_unload()
merged.save_pretrained("/kaggle/working/merged_model")
processor.save_pretrained("/kaggle/working/merged_model")
print("Merged model saved!")

In [ ]:
# Quick test
from transformers import AutoModelForImageTextToText

test_model = AutoModelForImageTextToText.from_pretrained(
    "/kaggle/working/merged_model", dtype=torch.bfloat16, device_map="auto"
)
test_model.eval()

test_images = !ls /kaggle/working/images/Images_train/ | shuf | head -4

for img_name in test_images:
    img = Image.open(f"/kaggle/working/images/Images_train/{img_name}").convert("RGB")
    conv = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [{"type": "image", "image": img}, {"type": "text", "text": USER_PROMPT}]},
    ]
    inputs = processor.apply_chat_template(conv, add_generation_prompt=True, return_tensors="pt", return_dict=True, tokenize=True).to(test_model.device)
    with torch.inference_mode():
        out = test_model.generate(**inputs, max_new_tokens=256, do_sample=False)
    response = processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    print(f"\n{img_name}: {response[:200]}")

In [ ]:
# Upload to HuggingFace
from huggingface_hub import HfApi

api = HfApi()
api.upload_folder(
    folder_path="/kaggle/working/merged_model",
    repo_id="marcelo-earth/LFM2.5-VL-450M-satellite-triage",
    repo_type="model",
    commit_message="Exp 4: balanced MEDIUM vs LOW+SKIP fine-tune (Kaggle T4)",
)
print("Uploaded!")